In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df = df[['Pclass','Sex','Age', 'Fare', 'Survived']]
df.head()

,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [4]:
df.isna().sum()

Pclass        0
Sex           0
Age         177
Fare          0
Survived      0
dtype: int64

In [5]:
df.shape

(891, 5)

In [6]:
df['Age'] = df.Age.fillna(df.Age.mean())
df.isna().sum()

Pclass      0
Sex         0
Age         0
Fare        0
Survived    0
dtype: int64

In [7]:
df.describe()

,Pclass,Age,Fare,Survived
count,891.000000,891.000000,891.000000,891.000000
mean,2.308642,29.699118,32.204208,0.383838
std,0.836071,13.002015,49.693429,0.486592
min,1.000000,0.420000,0.000000,0.000000
25%,2.000000,22.000000,7.910400,0.000000
50%,3.000000,29.699118,14.454200,0.000000
75%,3.000000,35.000000,31.000000,1.000000
max,3.000000,80.000000,512.329200,1.000000


In [8]:
df.Survived.value_counts()

0    549
1    342
Name: Survived, dtype: int64

In [9]:
# ratio to check imbalance
342/549  #not so imbalanced

0.6229508196721312

In [10]:
df['Sex'] =pd.get_dummies(df['Sex'], drop_first=True)
df['Sex']

0      1
1      0
2      0
3      0
4      1
      ..
886    1
887    0
888    0
889    1
890    1
Name: Sex, Length: 891, dtype: uint8

In [11]:
X= df.drop('Survived',axis='columns')
y=df.Survived

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[ 0.82737724,  0.73769513, -0.5924806 , -0.50244517],
       [-1.56610693, -1.35557354,  0.63878901,  0.78684529],
       [ 0.82737724, -1.35557354, -0.2846632 , -0.48885426],
       ...,
       [ 0.82737724, -1.35557354,  0.        , -0.17626324],
       [-1.56610693,  0.73769513, -0.2846632 , -0.04438104],
       [ 0.82737724,  0.73769513,  0.17706291, -0.49237783]])

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled,y,stratify=y,test_size=0.2, random_state=30)


In [16]:
X_train.shape

(712, 4)

In [17]:
X_test.shape

(179, 4)

In [20]:
y_train.value_counts()

0    439
1    273
Name: Survived, dtype: int64

In [21]:
# check ratio for imbalance
273/439   #similar to the previous data

0.621867881548975

In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

score =cross_val_score(DecisionTreeClassifier(),X,y,cv=5)
score

array([0.72067039, 0.78089888, 0.82022472, 0.78651685, 0.79213483])

In [24]:
score.mean()

0.7800891343920657

In [27]:
# let's use bagging classifier

from sklearn.ensemble import BaggingClassifier
bm = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)
scores = cross_val_score(bm,X,y,cv=5)
scores.mean()

C:\Users\pc\anaconda3\Lib\site-packages\sklearn\ensemble\_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
C:\Users\pc\anaconda3\Lib\site-packages\sklearn\ensemble\_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
C:\Users\pc\anaconda3\Lib\site-packages\sklearn\ensemble\_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
C:\Users\pc\anaconda3\Lib\site-packages\sklearn\ensemble\_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
C:\Users\pc\anaconda3\Lib\site-packages\sklearn\ensemble\_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


0.8069989328981231

we can see the boost in performance

In [28]:
from sklearn.ensemble import RandomForestClassifier
scr = cross_val_score(RandomForestClassifier(),X,y,cv=5)
scr.mean()

0.8114933149205952

In [29]:
# underneath the Randomforest it uses bagging method